# Learning Pytorch

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [2]:
training_data = datasets.FashionMNIST(
    root = "data",
    train = True,
    download = True,
    transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

In [3]:
test_data = datasets.FashionMNIST(
    root = "data",
    train = False,
    download = True,
    transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale = True)])
)

In [4]:
batch_size = 64

train_dataloader = DataLoader(training_data, batch_size = batch_size)
test_dataloader = DataLoader(test_data, batch_size = batch_size)

In [5]:
for X, y in test_dataloader:
    print(f"shape of X: [N, C, H, W]: {X.shape}")
    print(f"shape of Y: {y.shape} {y.dtype}")
    break

shape of X: [N, C, H, W]: torch.Size([64, 1, 28, 28])
shape of Y: torch.Size([64]) torch.int64


In [6]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"using: {device}")

using: mps


In [15]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512,512),
            nn.LeakyReLU(),
            nn.Linear(512, 512),
            nn.LeakyReLU(),
            nn.Linear(512, 10)
        )
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


In [16]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): LeakyReLU(negative_slope=0.01)
    (4): Linear(in_features=512, out_features=512, bias=True)
    (5): LeakyReLU(negative_slope=0.01)
    (6): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [17]:
# Loss Function
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [18]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")

In [19]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


In [26]:
epochs = 10

for t in range(epochs):
    print(f"Epochs: {t+1}\n-----------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("done!")

Epochs: 1
-----------------------------
loss: 0.286722 [   64/60000]
loss: 0.474869 [ 6464/60000]
loss: 0.278897 [12864/60000]
loss: 0.481331 [19264/60000]
loss: 0.412208 [25664/60000]
loss: 0.414788 [32064/60000]
loss: 0.435882 [38464/60000]
loss: 0.650243 [44864/60000]
loss: 0.560767 [51264/60000]
loss: 0.412334 [57664/60000]
Test Error: 
 Accuracy: 83.6%, Avg loss: 0.464303 

Epochs: 2
-----------------------------
loss: 0.283981 [   64/60000]
loss: 0.472922 [ 6464/60000]
loss: 0.277290 [12864/60000]
loss: 0.479929 [19264/60000]
loss: 0.408690 [25664/60000]
loss: 0.413337 [32064/60000]
loss: 0.434184 [38464/60000]
loss: 0.648326 [44864/60000]
loss: 0.558903 [51264/60000]
loss: 0.411023 [57664/60000]
Test Error: 
 Accuracy: 83.6%, Avg loss: 0.462882 

Epochs: 3
-----------------------------
loss: 0.281316 [   64/60000]
loss: 0.471005 [ 6464/60000]
loss: 0.275738 [12864/60000]
loss: 0.478596 [19264/60000]
loss: 0.405377 [25664/60000]
loss: 0.411856 [32064/60000]
loss: 0.432567 [38464/

In [ ]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

In [ ]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only = True))

In [ ]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')